In [1]:
import random

n=10
k=10

elements = list(range(n))
elements

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [8]:
import numpy as np

def echantillon(n):
    elements = list(range(n))
    tirage1 = random.choices(elements, k=n)
    return np.unique(tirage1).tolist()
    

In [44]:
list_ = []
for _ in range(5):
    ids = echantillon(50)
    list_ += ids

np.unique(list_).shape

(49,)

In [45]:
tab = np.array([[1,2,3],[4,5,6],[7,8,9]])

In [49]:
tab[[0,2]]

array([[1, 2, 3],
       [7, 8, 9]])

In [52]:
ids = [0,2]
tab2 = np.array([ tab[:,id] for id in ids ])

In [53]:
tab2

array([[1, 4, 7],
       [3, 6, 9]])

In [6]:
!pip install kahypar

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 487.5 kB/s eta 0:00:0000:0100:01


In [11]:
from utils.ClusterEnsembles.ClusterEnsembles import mcla


label1 = np.array([1, 1, 1, 2, 2, 3, 3])

label2 = np.array([2, 2, 2, 3, 3, 1, 1])

label3 = np.array([4, 4, 2, 2, 3, 3, 3])

label4 = np.array([1, 2, np.nan, 1, 2, np.nan, np.nan]) # `np.nan`: missing value

labels = np.array([label1, label2, label3, label4])

mcla(labels,2,0)

array([1, 1, 1, 0, 0, 0, 0])

In [1]:
from datasets.load import load_msrcv1

dataset = load_msrcv1("datasets/")

In [5]:
dataset["X"][0].shape

(24, 210)

In [2]:
from bagging import bagging_prime

r = bagging_prime(dataset["X"], 7)

/home/mbe/Bureau/theme/memoire/workflows/bagging.py:43: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if typeweak is "mcles":
/home/mbe/anaconda3/lib/python3.10/site-packages/qpsolvers/conversions/ensure_sparse_matrices.py:38: UserWarning: Converted P to scipy.sparse.csc.csc_matrix
For best performance, build P as a scipy.sparse.csc_matrix rather than as a numpy.ndarray
  warnings.warn(
/home/mbe/anaconda3/lib/python3.10/site-packages/qpsolvers/conversions/ensure_sparse_matrices.py:38: UserWarning: Converted A to scipy.sparse.csc.csc_matrix
For best performance, build A as a scipy.sparse.csc_matrix rather than as a numpy.ndarray
  warnings.warn(
/home/mbe/Bureau/theme/memoire/workflows/bagging.py:43: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if typeweak is "mcles":
/home/mbe/Bureau/theme/memoire/workflows/bagging.py:43: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if typeweak is "mcles":


IndexError: list index out of range

In [233]:
import numpy as np
import networkx as nx
from bokeh.io import show, output_notebook
from bokeh.models import (BoxZoomTool, Circle, Column, CustomJS, HoverTool,
                          MultiLine, Plot, Range1d, RangeSlider, ResetTool,
                          TapTool, WheelZoomTool)
from bokeh.plotting import figure, from_networkx

def calc_color(v, colors_clusters, mol):
    return v["_color"] if v["_type"] == "spin" else colors_clusters[mol]
        
def visualize_graph_madbyte(graph, colors_clusters = None):

    # Calcul des atributs de visualisations des aretes
    try:
        max_weight = max(nx.get_edge_attributes(graph, 'weight').values())
    except ValueError:
        max_weight = 1.0
    calc_alpha = lambda x: 0.1 + 0.6 * (x / max_weight)
    edge_attrs = {(s,e): calc_alpha(d['weight']) for s,e,d in graph.edges(data=True)}
    nx.set_edge_attributes(graph, edge_attrs, "_alpha")
    
    # Calcul des atributs de visualisation des noeuds
    node_attrs = {
        k: {
            "_color" : v["_color"] if colors_clusters == None else calc_color(v, colors_clusters, k),
            "_num_members": len(eval(v.get('members', '[]')))
        }
        for k,v in graph.nodes(data=True) }
    nx.set_node_attributes(graph, node_attrs)
    
    #print(graph.nodes(data=True)["HND_Azithromycin"])
    # Créer une figure Bokeh
    plot = figure(title="Interactive Graph", tools="pan,wheel_zoom,box_zoom,reset,save",x_range=Range1d(-1.1, 1.1), y_range=Range1d(-1.1, 1.1))
    
    # Convertir le graphe NetworkX en un graphe Bokeh
    plot_graph = from_networkx(graph, nx.spring_layout, scale=1, center=(0, 0))
    plot_graph.node_renderer.glyph = Circle(size='_size', fill_color="_color")
    plot_graph.node_renderer.selection_glyph = Circle(size=15, fill_color="_color")#works
    plot_graph.node_renderer.hover_glyph = Circle(size=15, fill_color="red")#works
    plot_graph.edge_renderer.glyph = MultiLine(line_alpha="_alpha", line_width=1)
    
    # Ajouter le graphe à la figure
    plot.renderers.append(plot_graph)

    # Ajouter un outil de survol pour afficher les informations des nœuds
    #hover = HoverTool(tooltips=[("Node", "@index"), ("Type", "@_type"), ("Members", "@members"), ("Nombre de membre", "@_num_members")])
    hover = HoverTool(tooltips=[("Node", "@index"), ("Members", "@members")])
    plot.add_tools(hover)

    # Ajouter d'autres outils d'interaction
    plot.add_tools(BoxZoomTool(), ResetTool())

    # Afficher la figure dans le notebook ou dans une fenêtre séparée
    output_notebook()
    show(plot)


In [234]:
import networkx as nx

G = nx.read_graphml("./test_similarity_network_network.graphml")

In [235]:
for component in nx.connected_components(G):
    print(component)

{'HND_Thiamphenicol_0', 'HND_Thiamphenicol', 'HND_Chloramphenicol_0', 'HND_Chloramphenicol'}
{'HND_Erythromycin_1', 'HND_Erythromycin_3', 'HND_Erythromycin_0', 'HND_Roxithromycin_0', 'HND_Azithromycin', 'HND_Roxithromycin_2', 'HND_Roxithromycin', 'HND_Roxithromycin_3', 'HND_Roxithromycin_1', 'HND_Erythromycin', 'HND_Azithromycin_1', 'HND_Azithromycin_2', 'HND_Erythromycin_5', 'HND_Azithromycin_0', 'HND_Roxithromycin_5', 'HND_Roxithromycin_4'}


In [236]:
visualize_graph_madbyte(G)

Loading BokehJS ...

In [237]:
G = nx.read_graphml("./test_association_network_all.graphml")

clusters_mol = {}
k=0
for component in nx.connected_components(G):
    for node in component:
        if G.nodes[node]["_type"] == "standard":
            clusters_mol[node] = k
    k = k + 1

clusters_mol

{'HND_Thiamphenicol': 0,
 'HND_Chloramphenicol': 0,
 'HND_Erythromycin': 1,
 'HND_Azithromycin': 1,
 'HND_Roxithromycin': 1}

In [238]:
import random
import matplotlib.colors as mcolors

def brightness(str_color):
    pr, pg, pb = mcolors.to_rgb("#D7CE5F")
    return (pr + pg + pb)/3

def generate_random_colors(n, exclude_colors=[], minb=0.3, maxb=0.7):
    # Obtenir la liste des couleurs de base
    i = 0
    colors = []
    while i < n:
        color = "#" + "".join([random.choice("0123456789ABCDEF") for j in range(6)])
        if color not in exclude_colors:
            b = brightness(color)
            if b > minb and b < maxb :
                colors.append(color)
                i += 1
            
    return colors

def colors_clusters(K, clusters_mol, exclude_colors=[]):
    colors = generate_random_colors(K, exclude_colors=exclude_colors)
    res = { mol : colors[idcluster]  for   mol, idcluster in clusters_mol.items()}
    return res

In [239]:
colors = colors_clusters(2, clusters_mol, exclude_colors=["#0ffbff", "#d3d7cf"])

In [240]:
colors

{'HND_Thiamphenicol': '#C272B4',
 'HND_Chloramphenicol': '#C272B4',
 'HND_Erythromycin': '#11E5BD',
 'HND_Azithromycin': '#11E5BD',
 'HND_Roxithromycin': '#11E5BD'}

In [241]:
visualize_graph_madbyte(G, colors_clusters=colors)

Loading BokehJS ...

In [6]:
from numpy import zeros, nan

names_mols = ['HND_Chloramphenicol','HND_Thiamphenicol','HND_Azithromycin','HND_Erythromycin','HND_Roxithromycin',"r"]

n = len(names_mols)
mols = zeros(n)
for i in range(n):
    try:
        mols[i] = clusters_mol[names_mols[i]]
    except :
        mols[i] = nan
mols

array([ 0.,  0.,  1.,  1.,  1., nan])

In [7]:
G = nx.read_graphml("./test_hybrid_network.graphml")
for component in nx.connected_components(G):
    print(component)

{'HND_Thiamphenicol', 'HND_Chloramphenicol_0/HND_Thiamphenicol_0', 'HND_Chloramphenicol'}
{'HND_Roxithromycin_3/HND_Azithromycin_1', 'HND_Azithromycin', 'HND_Roxithromycin', 'HND_Roxithromycin_2/HND_Erythromycin_3', 'HND_Erythromycin_5/HND_Azithromycin_2/HND_Roxithromycin_4', 'HND_Erythromycin', 'HND_Erythromycin_3/HND_Azithromycin_1', 'HND_Roxithromycin_2/HND_Azithromycin_1', 'HND_Azithromycin_0/HND_Roxithromycin_0/HND_Erythromycin_0'}
